# Ch02-01: 이산 마르코프 체인


## 학습 목표

- 전이 행렬의 성질과 n-단계 전이 확률
- 정상 분포의 존재와 유일성 조건
- 에르고딕 정리와 수렴 속도
- PageRank 알고리즘의 마르코프 체인 해석


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
np.random.seed(42)

## 이론적 배경

### 마르코프 체인 기초
상태 공간 $S = \{1, 2, \ldots, m\}$ 위의 확률 과정 $\{X_n\}$이 **마르코프 성질**을 만족하면:
$$P(X_{n+1} = j | X_n = i, X_{n-1}, \ldots, X_0) = P(X_{n+1} = j | X_n = i) = p_{ij}$$

**전이 행렬** $\mathbf{P} = [p_{ij}]$는 행 합이 1인 확률 행렬입니다.
$n$-단계 전이 확률: $p_{ij}^{(n)} = (\mathbf{P}^n)_{ij}$ (Chapman-Kolmogorov 방정식)

### 정상 분포 (Stationary Distribution)
벡터 $\boldsymbol{\pi}$가 $\boldsymbol{\pi} = \boldsymbol{\pi} \mathbf{P}$를 만족하고 $\sum_i \pi_i = 1$이면 정상 분포입니다.
**에르고딕 정리**: 비가약(irreducible), 비주기(aperiodic) 체인은 유일한 정상 분포를 가지며,
$$\lim_{n\to\infty} p_{ij}^{(n)} = \pi_j \quad \forall i$$

### PageRank
$$\pi_j = \frac{1-d}{N} + d \sum_{i \to j} \frac{\pi_i}{L(i)}$$
여기서 $d$는 감쇠 인자, $L(i)$는 페이지 $i$의 아웃링크 수입니다.


## 구현 가이드

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 날씨 마르코프 체인 (맑음, 흐림, 비)
P = np.array([
    [0.7, 0.2, 0.1],
    [0.3, 0.4, 0.3],
    [0.2, 0.3, 0.5]
])
states = ['맑음', '흐림', '비']

# n-단계 전이 확률
P_n = np.linalg.matrix_power(P, 50)
print("50-단계 전이 행렬 (각 행이 동일 = 수렴):")
print(P_n.round(4))

# 정상 분포: 고유값 분해로 구하기
eigenvalues, eigenvectors = np.linalg.eig(P.T)
idx = np.argmin(np.abs(eigenvalues - 1))
pi = np.real(eigenvectors[:, idx])
pi = pi / pi.sum()
print(f"\n정상 분포: {dict(zip(states, pi.round(4)))}")

# 시뮬레이션으로 검증
n_steps = 10000
state = 0
counts = np.zeros(3)
for _ in range(n_steps):
    state = np.random.choice(3, p=P[state])
    counts[state] += 1
empirical = counts / n_steps
print(f"경험적 분포: {dict(zip(states, empirical.round(4)))}")

# 수렴 시각화
distributions = []
dist = np.array([1.0, 0.0, 0.0])
for n in range(30):
    distributions.append(dist.copy())
    dist = dist @ P

distributions = np.array(distributions)
fig, ax = plt.subplots(figsize=(10, 6))
for i, s in enumerate(states):
    ax.plot(distributions[:, i], label=s)
    ax.axhline(pi[i], color=f'C{i}', linestyle='--', alpha=0.5)
ax.set_xlabel('단계')
ax.set_ylabel('확률')
ax.set_title('마르코프 체인 수렴')
ax.legend()
plt.show()

---
## 연습 문제

### 문제 1 [★]

전이 행렬의 고유값을 분석하여 수렴 속도를 결정하세요. 두 번째로 큰 고유값의 절대값 $|\lambda_2|$가 **스펙트럼 갭**과 수렴 속도를 어떻게 결정하는지 분석하세요.

In [ ]:
# 스펙트럼 갭과 수렴 속도
P = np.array([[0.7, 0.2, 0.1], [0.3, 0.4, 0.3], [0.2, 0.3, 0.5]])

# TODO: 고유값 계산
# TODO: 스펙트럼 갭 = 1 - |lambda_2|
# TODO: mixing time 추정
# TODO: 다양한 P에 대해 비교


### 문제 2 [★★]

PageRank 알고리즘을 Power Iteration으로 구현하세요. 6개 웹페이지의 링크 구조를 정의하고 순위를 계산하세요. 감쇠 인자 $d$에 따른 순위 변화를 분석하세요.

In [ ]:
# PageRank 구현
def pagerank(adj_matrix, d=0.85, tol=1e-8):
    # TODO: Google 행렬 구성
    # TODO: Power iteration
    pass

# 6-노드 그래프
adj = np.array([
    [0,1,1,0,0,0],
    [0,0,1,1,0,0],
    [1,0,0,0,0,0],
    [0,0,0,0,1,1],
    [0,0,0,1,0,1],
    [0,0,0,1,0,0]
])


### 문제 3 [★★★]

흡수 마르코프 체인(absorbing chain)의 기본 행렬 $N = (I - Q)^{-1}$을 구하고, 흡수까지의 기대 단계 수와 흡수 확률을 계산하세요. 도박꾼의 파산 문제에 적용하세요.

In [ ]:
# 흡수 마르코프 체인 - 도박꾼의 파산
# 초기 자본 i, 목표 N, 단판 승률 p
# TODO: 전이 행렬 구성 (상태 0과 N은 흡수 상태)
# TODO: 기본 행렬 N 계산
# TODO: 흡수까지 기대 단계 수
# TODO: 파산 확률 (이론값과 비교)


---
## 참고 자료

- Norris, J.R. (1997). 'Markov Chains'
- Brin, S. & Page, L. (1998). 'The Anatomy of a Large-Scale Hypertextual Web Search Engine'
